### Connect to Database Code

In [2]:
DECLARE @co_dpd_days INT = 120;

DROP TABLE IF EXISTS [ConsumerLoans].[dbo].[Quantum_TermLoan_Dynamic];

WITH raw_term AS (
    SELECT
        ACCOUNTID AS ACCOUNT_ID,
        SNAPSHOTDATE AS SNAPSHOT_DATE,
        MOC AS MOC,
        COMMITTEDQUARTERNAME AS COMMITTED_QUARTER_NAME,
        BOOKEDAMOUNT AS BOOKED_AMOUNT,
        PRODUCTTYPE AS PRODUCT_TYPE,
        LOANSTATUS AS LOAN_STATUS,
        UPB AS UPB,
        FUNDEDAMOUNTMTD AS FUNDED_AMOUNT_MTD,
        FACILITYSIZECURRENT AS FACILITY_SIZE_CURRENT,
        NUMBEROFDRAWSMTD AS NUMBER_OF_DRAWS_MTD,
        SCHEDULEDPAYMENTAMOUNTMTD AS SCHEDULED_PAYMENT_AMOUNT_MTD,
        PAIDPRINCIPALINTERESTAMOUNTMTD AS PAID_PRINCIPAL_INTEREST_AMOUNT_MTD,
        PAIDPRINCIPALAMOUNTMTD AS PAID_PRINCIPAL_AMOUNT_MTD,
        PAIDINTERESTAMOUNTMTD AS PAID_INTEREST_AMOUNT_MTD,
        PAYMENTSPERMONTH AS PAYMENTS_PER_MONTH,
        DAYSPASTDUE AS DAYS_PAST_DUE,
        CHARGEOFFAMOUNTMTD AS CHARGEOFF_AMOUNT_MTD,
        RECOVERYAMOUNTMTD AS RECOVERY_AMOUNT_MTD,
        LOCSTATUS AS LOC_STATUS,
        DRAWFEESMTD AS DRAW_FEES_MTD,
        DRAWFEESLTD AS DRAW_FEES_LTD,
        PAIDLATEFEESMTD AS PAID_LATE_FEES_MTD,
        PAIDLOCINACTIVITYFEESMTD AS PAID_LOC_INACTIVITY_FEES_MTD,
        PAIDCOMMITMENTFEESMTD AS PAID_COMMITMENT_FEES_MTD,
        LASTLOCKEDDATE AS LAST_LOCKED_DATE
    FROM [ConsumerLoans].[dbo].[Quantum_Dynamic]
    WHERE PRODUCTTYPE = 'Term Loan'
),
first_co AS (
    SELECT
        ACCOUNT_ID,
        MIN(SNAPSHOT_DATE) AS first_co_date
    FROM raw_term
    WHERE DAYS_PAST_DUE > @co_dpd_days
       OR LOAN_STATUS = 'ChargeOff'
       OR LOC_STATUS = 'ChargedOff'
    GROUP BY ACCOUNT_ID
),
base AS (
    SELECT
        r.*,
        f.first_co_date
    FROM raw_term r
    LEFT JOIN first_co f
        ON r.ACCOUNT_ID = f.ACCOUNT_ID
)
SELECT
    *
INTO [ConsumerLoans].[dbo].[Quantum_TermLoan_Dynamic]
FROM base;

ALTER TABLE [ConsumerLoans].[dbo].[Quantum_TermLoan_Dynamic]
ADD
	bom_bal FLOAT NULL,
    eom_bal FLOAT NULL,
    balance_clean_note VARCHAR(100) NULL,
    funded_amount_mtd_clean FLOAT NULL,
    funded_amount_mtd_clean_note VARCHAR(100) NULL,
    paid_principal_interest_amount_mtd_clean FLOAT NULL,
    paid_principal_amount_mtd_clean FLOAT NULL,
    paid_interest_amount_mtd_clean FLOAT NULL,
    payment_clean_note VARCHAR(200) NULL,
    first_co INT NULL,
    has_been_co INT NULL,
    chargeoff_amount_mtd_clean FLOAT NULL,
    chargeoff_amount_mtd_clean_note VARCHAR(100) NULL,
    cumulative_chargeoff_amount FLOAT NULL,
    recovery_amount_mtd_clean FLOAT NULL,
    recovery_amount_mtd_clean_note VARCHAR(100) NULL,
    cumulative_recovery_amount FLOAT NULL,
--    reported_co INT NULL,
--    dpd120_co INT NULL,
    co_flag_source VARCHAR(50) NULL;
GO

DECLARE @co_dpd_days INT = 120;
DECLARE @funding_moc_grace_months INT = 2;
UPDATE q
SET
    first_co =
        CASE
            WHEN q.SNAPSHOT_DATE = q.first_co_date THEN 1
            ELSE 0
        END,

    has_been_co =
        CASE
            WHEN q.first_co_date IS NOT NULL
             AND q.SNAPSHOT_DATE >= q.first_co_date THEN 1
            ELSE 0
        END,
/*
    reported_co =
        CASE
            WHEN q.LOAN_STATUS = 'ChargeOff'
              OR q.LOC_STATUS = 'ChargedOff'
              OR COALESCE(q.CHARGEOFF_AMOUNT_MTD, 0) > 0
            THEN 1 ELSE 0
        END,

    dpd120_co =
        CASE
            WHEN q.DAYS_PAST_DUE > @co_dpd_days
            THEN 1 ELSE 0
        END,
*/
    co_flag_source =
        CASE
            WHEN (
                    q.LOAN_STATUS = 'ChargeOff'
                 OR q.LOC_STATUS = 'ChargedOff'
                 OR COALESCE(q.CHARGEOFF_AMOUNT_MTD, 0) > 0
                 )
             AND q.DAYS_PAST_DUE > @co_dpd_days
            THEN 'Both reported CO and DPD > 120'

            WHEN q.LOAN_STATUS = 'ChargeOff'
              OR q.LOC_STATUS = 'ChargedOff'
              OR COALESCE(q.CHARGEOFF_AMOUNT_MTD, 0) > 0
            THEN 'Reported CO'

            WHEN q.DAYS_PAST_DUE > @co_dpd_days
            THEN 'DPD > 120'

            ELSE NULL
        END
FROM [ConsumerLoans].[dbo].[Quantum_TermLoan_Dynamic] q;

WITH funding_candidates AS (
    SELECT
        *,
        CASE
            WHEN FUNDED_AMOUNT_MTD IS NULL
             AND COALESCE(MOC, @funding_moc_grace_months + 1) BETWEEN 0 AND @funding_moc_grace_months
             AND COALESCE(UPB, 0) > 0
             AND COALESCE(BOOKED_AMOUNT, 0) > 0
            THEN 1 ELSE 0
        END AS missing_early_funding_candidate
    FROM [ConsumerLoans].[dbo].[Quantum_TermLoan_Dynamic]
),
x AS (
    SELECT
        *,

        MIN(CASE
                WHEN COALESCE(FUNDED_AMOUNT_MTD, 0) > 0
                  OR missing_early_funding_candidate = 1
                THEN SNAPSHOT_DATE
            END) OVER (
            PARTITION BY ACCOUNT_ID
        ) AS first_funding_date,

        MIN(CASE
                WHEN COALESCE(FUNDED_AMOUNT_MTD, 0) > 0
                  OR missing_early_funding_candidate = 1
                THEN MOC
            END) OVER (
            PARTITION BY ACCOUNT_ID
        ) AS first_funding_moc,

        LAG(UPB) OVER (
            PARTITION BY ACCOUNT_ID
            ORDER BY SNAPSHOT_DATE
        ) AS prior_upb
    FROM funding_candidates
),
counted AS (
    SELECT
        *,

        SUM(CASE
                WHEN COALESCE(FUNDED_AMOUNT_MTD, 0) > 0
                  OR (
                        missing_early_funding_candidate = 1
                    AND SNAPSHOT_DATE = first_funding_date
                     )
                THEN 1 ELSE 0
            END) OVER (
            PARTITION BY ACCOUNT_ID
            ORDER BY SNAPSHOT_DATE
            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
        ) AS funding_count_to_date
    FROM x
),
cleaned AS (
    SELECT
        ACCOUNT_ID,
        SNAPSHOT_DATE,

        CASE
            -- Missing first funding near origination: impute to booked amount when UPB confirms funding.
            WHEN FUNDED_AMOUNT_MTD IS NULL
             AND missing_early_funding_candidate = 1
             AND SNAPSHOT_DATE = first_funding_date
            THEN BOOKED_AMOUNT

            -- First funding month: cap to booked amount if raw funded is too high.
            WHEN SNAPSHOT_DATE = first_funding_date
             AND COALESCE(FUNDED_AMOUNT_MTD, 0) > COALESCE(BOOKED_AMOUNT, 0)
             AND COALESCE(BOOKED_AMOUNT, 0) > 0
            THEN BOOKED_AMOUNT


            -- Later repeated funding records: keep if UPB increased; otherwise zero out.
            WHEN COALESCE(FUNDED_AMOUNT_MTD, 0) > 0
             AND funding_count_to_date > 1
             AND prior_upb IS NOT NULL
             AND COALESCE(UPB, 0) > COALESCE(prior_upb, 0)
            THEN COALESCE(FUNDED_AMOUNT_MTD, 0)

            WHEN COALESCE(FUNDED_AMOUNT_MTD, 0) > 0 
             AND funding_count_to_date > 1
            THEN 0

            ELSE COALESCE(FUNDED_AMOUNT_MTD, 0)
        END AS funded_amount_mtd_clean,

        CASE
            WHEN FUNDED_AMOUNT_MTD IS NULL
             AND missing_early_funding_candidate = 1
             AND SNAPSHOT_DATE = first_funding_date
            THEN 'Missing early funding imputed to booked amount'

            WHEN SNAPSHOT_DATE = first_funding_date
             AND COALESCE(FUNDED_AMOUNT_MTD, 0) > COALESCE(BOOKED_AMOUNT, 0)
             AND COALESCE(BOOKED_AMOUNT, 0) > 0
            THEN 'First funding capped to booked amount'

            WHEN COALESCE(FUNDED_AMOUNT_MTD, 0) > 0
             AND funding_count_to_date > 1
             AND prior_upb IS NOT NULL
             AND COALESCE(UPB, 0) > COALESCE(prior_upb, 0)
            THEN 'Repeated funding kept; UPB increased'

            WHEN COALESCE(FUNDED_AMOUNT_MTD, 0) > 0
             AND funding_count_to_date > 1
            THEN 'Repeated funding zeroed'

            ELSE NULL
        END AS funded_amount_mtd_clean_note
    FROM counted
)
UPDATE q
SET
    funded_amount_mtd_clean = c.funded_amount_mtd_clean,
    funded_amount_mtd_clean_note = c.funded_amount_mtd_clean_note
FROM [ConsumerLoans].[dbo].[Quantum_TermLoan_Dynamic] q
JOIN cleaned c
    ON q.ACCOUNT_ID = c.ACCOUNT_ID
   AND q.SNAPSHOT_DATE = c.SNAPSHOT_DATE;

WITH balance_seed AS (
    SELECT
        *,
        COALESCE(funded_amount_mtd_clean, 0) - COALESCE(PAID_PRINCIPAL_AMOUNT_MTD, 0) AS funding_activity_bal,
        LEAD(LOAN_STATUS) OVER (
            PARTITION BY ACCOUNT_ID
            ORDER BY SNAPSHOT_DATE
        ) AS next_loan_status,
        LEAD(UPB) OVER (
            PARTITION BY ACCOUNT_ID
            ORDER BY SNAPSHOT_DATE
        ) AS next_upb,
        LAG(
            CASE
                WHEN has_been_co = 1 THEN 0
                ELSE COALESCE(UPB, 0)
            END
        ) OVER (
            PARTITION BY ACCOUNT_ID
            ORDER BY SNAPSHOT_DATE
        ) AS prior_reported_eom,
        MIN(CASE WHEN COALESCE(funded_amount_mtd_clean, 0) > 0 THEN SNAPSHOT_DATE END) OVER (
            PARTITION BY ACCOUNT_ID
        ) AS first_clean_funding_date,
        MAX(CASE
                WHEN LOAN_STATUS = 'Performing'
                  OR COALESCE(UPB, 0) > 0
                THEN 1
                ELSE 0
            END) OVER (
            PARTITION BY ACCOUNT_ID
            ORDER BY SNAPSHOT_DATE
            ROWS BETWEEN 1 FOLLOWING AND UNBOUNDED FOLLOWING
        ) AS has_future_performing_or_balance
    FROM [ConsumerLoans].[dbo].[Quantum_TermLoan_Dynamic]
),
balance_base AS (
    SELECT
        ACCOUNT_ID,
        SNAPSHOT_DATE,
        has_been_co,
        funding_activity_bal,
        first_clean_funding_date,
        has_future_performing_or_balance,
        CASE
            WHEN has_been_co = 1 THEN 0

            WHEN UPB IS NULL
             AND LOAN_STATUS = 'Closed'
             AND COALESCE(funded_amount_mtd_clean, 0) > 0
             AND SNAPSHOT_DATE = first_clean_funding_date
             AND COALESCE(next_upb, 0) > 0
             AND funding_activity_bal > 0
            THEN funding_activity_bal

            WHEN UPB IS NULL
             AND LOAN_STATUS = 'Closed'
             AND next_loan_status = 'Performing'
             AND COALESCE(prior_reported_eom, 0) > 0
             AND COALESCE(PAID_PRINCIPAL_AMOUNT_MTD, 0) > 0
             AND COALESCE(prior_reported_eom, 0)
                 + COALESCE(funded_amount_mtd_clean, 0)
                 - COALESCE(PAID_PRINCIPAL_AMOUNT_MTD, 0) > 0
            THEN COALESCE(prior_reported_eom, 0)
                 + COALESCE(funded_amount_mtd_clean, 0)
                 - COALESCE(PAID_PRINCIPAL_AMOUNT_MTD, 0)

            ELSE COALESCE(UPB, 0)
        END AS eom_bal,
        CASE
            WHEN UPB IS NULL
             AND LOAN_STATUS = 'Closed'
             AND COALESCE(funded_amount_mtd_clean, 0) > 0
             AND SNAPSHOT_DATE = first_clean_funding_date
             AND COALESCE(next_upb, 0) > 0
             AND funding_activity_bal > 0
            THEN 'Opening closed/funded EOM inferred from funding activity'

            WHEN UPB IS NULL
             AND LOAN_STATUS = 'Closed'
             AND next_loan_status = 'Performing'
             AND COALESCE(prior_reported_eom, 0) > 0
             AND COALESCE(PAID_PRINCIPAL_AMOUNT_MTD, 0) > 0
             AND COALESCE(prior_reported_eom, 0)
                 + COALESCE(funded_amount_mtd_clean, 0)
                 - COALESCE(PAID_PRINCIPAL_AMOUNT_MTD, 0) > 0
            THEN 'Temporary closed EOM inferred from reported principal'

            ELSE NULL
        END AS eom_bal_note
    FROM balance_seed
),
balance_periods AS (
    SELECT
        ACCOUNT_ID,
        SNAPSHOT_DATE,
        eom_bal,
        eom_bal_note,
        LAG(
            eom_bal
        ) OVER (
            PARTITION BY ACCOUNT_ID
            ORDER BY SNAPSHOT_DATE
        ) AS prior_eom_bal,
        LAG(
            CASE
                WHEN has_been_co = 1 THEN 0
                ELSE funding_activity_bal
            END
        ) OVER (
            PARTITION BY ACCOUNT_ID
            ORDER BY SNAPSHOT_DATE
        ) AS prior_funding_activity_bal,
        first_clean_funding_date,
        has_future_performing_or_balance
    FROM balance_base
)
UPDATE q
SET
    eom_bal = b.eom_bal,
    bom_bal =
        CASE
            WHEN q.first_co = 1
            THEN COALESCE(b.prior_eom_bal, 0)

            WHEN q.has_been_co = 1
            THEN 0

            WHEN q.SNAPSHOT_DATE = b.first_clean_funding_date
            THEN 0

            WHEN b.prior_eom_bal IS NULL
            THEN 0

            WHEN COALESCE(b.prior_eom_bal, 0) = 0
             AND COALESCE(b.prior_funding_activity_bal, 0) > 0
             AND (
                    COALESCE(b.eom_bal, 0) > 0
                 OR COALESCE(b.has_future_performing_or_balance, 0) = 1
                 )
            THEN b.prior_funding_activity_bal

            ELSE b.prior_eom_bal
        END,
    balance_clean_note =
        CASE
            WHEN b.eom_bal_note IS NOT NULL
             AND q.SNAPSHOT_DATE = b.first_clean_funding_date
            THEN b.eom_bal_note + '; First funding BOM zero'

            WHEN b.eom_bal_note IS NOT NULL
            THEN b.eom_bal_note

            WHEN q.first_co = 1
            THEN 'First charge-off BOM set to prior EOM'

            WHEN q.has_been_co = 1
            THEN 'Post-charge-off balance zeroed'

            WHEN q.UPB IS NULL
            THEN 'Null UPB treated as zero EOM'

            WHEN q.SNAPSHOT_DATE = b.first_clean_funding_date
            THEN 'First funding BOM set to zero'

            WHEN b.prior_eom_bal IS NULL
            THEN 'First observed BOM set to zero'

            WHEN COALESCE(b.prior_eom_bal, 0) = 0
             AND COALESCE(b.prior_funding_activity_bal, 0) > 0
             AND (
                    COALESCE(b.eom_bal, 0) > 0
                 OR COALESCE(b.has_future_performing_or_balance, 0) = 1
                 )
            THEN 'BOM adjusted from prior funding activity'

            ELSE NULL
        END
FROM [ConsumerLoans].[dbo].[Quantum_TermLoan_Dynamic] q
JOIN balance_periods b
    ON q.ACCOUNT_ID = b.ACCOUNT_ID
   AND q.SNAPSHOT_DATE = b.SNAPSHOT_DATE;

WITH chargeoff_clean AS (
    SELECT
        ACCOUNT_ID,
        SNAPSHOT_DATE,
        CASE
            WHEN first_co = 1
            THEN COALESCE(bom_bal, 0)
            ELSE 0
        END AS chargeoff_amount_mtd_clean,
        CASE
            WHEN first_co = 1
            THEN 'Chargeoff set to BOM at first CO'

            WHEN COALESCE(CHARGEOFF_AMOUNT_MTD, 0) <> 0
            THEN 'Reported chargeoff zeroed outside first CO'

            ELSE NULL
        END AS chargeoff_amount_mtd_clean_note
    FROM [ConsumerLoans].[dbo].[Quantum_TermLoan_Dynamic]
)
UPDATE q
SET
    chargeoff_amount_mtd_clean = c.chargeoff_amount_mtd_clean,
    chargeoff_amount_mtd_clean_note = c.chargeoff_amount_mtd_clean_note
FROM [ConsumerLoans].[dbo].[Quantum_TermLoan_Dynamic] q
JOIN chargeoff_clean c
    ON q.ACCOUNT_ID = c.ACCOUNT_ID
   AND q.SNAPSHOT_DATE = c.SNAPSHOT_DATE;

WITH cumulative_chargeoff AS (
    SELECT
        ACCOUNT_ID,
        SNAPSHOT_DATE,
        SUM(COALESCE(chargeoff_amount_mtd_clean, 0)) OVER (
            PARTITION BY ACCOUNT_ID
            ORDER BY SNAPSHOT_DATE
            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
        ) AS cumulative_chargeoff_amount
    FROM [ConsumerLoans].[dbo].[Quantum_TermLoan_Dynamic]
)
UPDATE q
SET cumulative_chargeoff_amount = c.cumulative_chargeoff_amount
FROM [ConsumerLoans].[dbo].[Quantum_TermLoan_Dynamic] q
JOIN cumulative_chargeoff c
    ON q.ACCOUNT_ID = c.ACCOUNT_ID
   AND q.SNAPSHOT_DATE = c.SNAPSHOT_DATE;

WITH payment_context AS (
    SELECT
        *,
        COALESCE(bom_bal, 0)
        + COALESCE(funded_amount_mtd_clean, 0)
        - COALESCE(chargeoff_amount_mtd_clean, 0)
        - COALESCE(eom_bal, 0) AS implied_principal_amount,
        PAID_INTEREST_AMOUNT_MTD AS reported_interest_amount,
        MAX(CASE
                WHEN LOAN_STATUS = 'Performing'
                  OR COALESCE(UPB, 0) > 0
                THEN 1 ELSE 0
            END) OVER (
            PARTITION BY ACCOUNT_ID
            ORDER BY SNAPSHOT_DATE
            ROWS BETWEEN 1 FOLLOWING AND UNBOUNDED FOLLOWING
        ) AS has_future_performing_or_balance
    FROM [ConsumerLoans].[dbo].[Quantum_TermLoan_Dynamic]
),
payment_clean AS (
    SELECT
        ACCOUNT_ID,
        SNAPSHOT_DATE,
        reported_interest_amount,
        CASE
            WHEN has_been_co = 1 THEN 0

            WHEN LOAN_STATUS = 'Closed'
             AND COALESCE(eom_bal, 0) = 0
             AND COALESCE(funded_amount_mtd_clean, 0) = 0
             AND COALESCE(chargeoff_amount_mtd_clean, 0) = 0
             AND COALESCE(has_future_performing_or_balance, 0) = 0
             AND implied_principal_amount > 0.01
             AND (
                    PAID_PRINCIPAL_AMOUNT_MTD IS NULL
                 OR ABS(COALESCE(PAID_PRINCIPAL_AMOUNT_MTD, 0) - implied_principal_amount) > 0.01
                 )
            THEN implied_principal_amount

            WHEN COALESCE(PAID_PRINCIPAL_AMOUNT_MTD, 0) > 0
             AND ABS(COALESCE(eom_bal, 0)
                     - (COALESCE(bom_bal, 0)
                        + COALESCE(funded_amount_mtd_clean, 0)
                        - COALESCE(chargeoff_amount_mtd_clean, 0))) <= 0.01
            THEN 0

            WHEN ABS(COALESCE(PAID_PRINCIPAL_AMOUNT_MTD, 0) - implied_principal_amount) > 0.01
            THEN implied_principal_amount

            ELSE PAID_PRINCIPAL_AMOUNT_MTD
        END AS paid_principal_amount_mtd_clean,
        CASE
            WHEN has_been_co = 1
             AND (
                    COALESCE(PAID_PRINCIPAL_INTEREST_AMOUNT_MTD, 0) <> 0
                 OR COALESCE(PAID_PRINCIPAL_AMOUNT_MTD, 0) <> 0
                 OR COALESCE(PAID_INTEREST_AMOUNT_MTD, 0) <> 0
                 )
            THEN 'Payment moved to recovery after charge-off'

            WHEN LOAN_STATUS = 'Closed'
             AND COALESCE(eom_bal, 0) = 0
             AND COALESCE(funded_amount_mtd_clean, 0) = 0
             AND COALESCE(chargeoff_amount_mtd_clean, 0) = 0
             AND COALESCE(has_future_performing_or_balance, 0) = 0
             AND implied_principal_amount > 0.01
             AND (
                    PAID_PRINCIPAL_AMOUNT_MTD IS NULL
                 OR ABS(COALESCE(PAID_PRINCIPAL_AMOUNT_MTD, 0) - implied_principal_amount) > 0.01
                 )
            THEN 'Closed payoff principal imputed'

            WHEN COALESCE(PAID_PRINCIPAL_AMOUNT_MTD, 0) > 0
             AND ABS(COALESCE(eom_bal, 0)
                     - (COALESCE(bom_bal, 0)
                        + COALESCE(funded_amount_mtd_clean, 0)
                        - COALESCE(chargeoff_amount_mtd_clean, 0))) <= 0.01
            THEN 'Principal not reflected in UPB zeroed'

            WHEN ABS(COALESCE(PAID_PRINCIPAL_AMOUNT_MTD, 0) - implied_principal_amount) > 0.01
            THEN 'Principal adjusted to UPB movement'

            ELSE NULL
        END AS payment_clean_note
    FROM payment_context
)
UPDATE q
SET
    paid_principal_amount_mtd_clean = p.paid_principal_amount_mtd_clean,
    paid_interest_amount_mtd_clean =
        CASE
            WHEN q.has_been_co = 1 THEN 0
            ELSE p.reported_interest_amount
        END,
    paid_principal_interest_amount_mtd_clean =
        CASE
            WHEN q.has_been_co = 1 THEN 0
            ELSE COALESCE(p.reported_interest_amount, 0)
               + COALESCE(p.paid_principal_amount_mtd_clean, 0)
        END,
    payment_clean_note = p.payment_clean_note
FROM [ConsumerLoans].[dbo].[Quantum_TermLoan_Dynamic] q
JOIN payment_clean p
    ON q.ACCOUNT_ID = p.ACCOUNT_ID
   AND q.SNAPSHOT_DATE = p.SNAPSHOT_DATE;

WITH recovery_clean AS (
    SELECT
        ACCOUNT_ID,
        SNAPSHOT_DATE,
        CASE
            WHEN has_been_co = 1
            THEN COALESCE(NULLIF(RECOVERY_AMOUNT_MTD, 0), PAID_PRINCIPAL_INTEREST_AMOUNT_MTD, 0)
            ELSE COALESCE(RECOVERY_AMOUNT_MTD, 0)
        END AS recovery_amount_mtd_clean,
        CASE
            WHEN has_been_co = 1
             AND COALESCE(RECOVERY_AMOUNT_MTD, 0) = 0
             AND COALESCE(PAID_PRINCIPAL_INTEREST_AMOUNT_MTD, 0) <> 0
            THEN 'Post-CO payment reclassified as recovery'

            ELSE NULL
        END AS recovery_amount_mtd_clean_note
    FROM [ConsumerLoans].[dbo].[Quantum_TermLoan_Dynamic]
)
UPDATE q
SET
    recovery_amount_mtd_clean = r.recovery_amount_mtd_clean,
    recovery_amount_mtd_clean_note = r.recovery_amount_mtd_clean_note
FROM [ConsumerLoans].[dbo].[Quantum_TermLoan_Dynamic] q
JOIN recovery_clean r
    ON q.ACCOUNT_ID = r.ACCOUNT_ID
   AND q.SNAPSHOT_DATE = r.SNAPSHOT_DATE;

WITH cumulative_recovery AS (
    SELECT
        ACCOUNT_ID,
        SNAPSHOT_DATE,
        SUM(COALESCE(recovery_amount_mtd_clean, 0)) OVER (
            PARTITION BY ACCOUNT_ID
            ORDER BY SNAPSHOT_DATE
            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
        ) AS cumulative_recovery_amount
    FROM [ConsumerLoans].[dbo].[Quantum_TermLoan_Dynamic]
)
UPDATE q
SET cumulative_recovery_amount = r.cumulative_recovery_amount
FROM [ConsumerLoans].[dbo].[Quantum_TermLoan_Dynamic] q
JOIN cumulative_recovery r
    ON q.ACCOUNT_ID = r.ACCOUNT_ID
   AND q.SNAPSHOT_DATE = r.SNAPSHOT_DATE;

(371325 rows affected)

(371325 rows affected)
(371325 rows affected)
(371325 rows affected)
(371325 rows affected)
(371325 rows affected)
(371325 rows affected)
(371325 rows affected)
(371325 rows affected)

Total execution time: 00:00:18.341